# Detecting AI-generated images with Defactify

This notebook trains and evaluates three models to distinguish real photographs from fully AI-generated images using the 96,000-image Defactify dataset.


## Setup

Set the dataset paths, random seed, training parameters, and execution switches. Select the available PyTorch device and configure offline loading when downloads are disabled.


In [4]:
import os
from pathlib import Path

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "README.md").exists() and (path / "environment.yml").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the project root.")
DATA_DIR = PROJECT_ROOT / "data"
HF_HOME = DATA_DIR / "huggingface"
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_HOME / "datasets")
os.environ["TORCH_HOME"] = str(DATA_DIR / "torch")

DATASET_ID = "Rajarshi-Roy-research/Defactify_Image_Dataset"
DATASET_REVISION = "1c3b714c9691eb6dd11c2c1e92eab1ac29584fa0"
EXPECTED_SIZES = {"train": 42_000, "validation": 9_000, "test": 45_000}
SOURCE_NAMES = {
    0: "real",
    1: "SD 2.1",
    2: "SDXL",
    3: "SD 3",
    4: "DALL-E 3",
    5: "Midjourney v6",
}
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-4
AUX_LOSS_WEIGHT = 0.2
RUN_LOAD_DATASET = True
RUN_DOWNLOAD = False
RUN_AUDIT = True
RUN_TRAIN = False
RUN_EXPORT_FINAL = False
RUN_FINAL_TEST = False

# Set before importing datasets/huggingface_hub. Restart the kernel after changing mode.
if RUN_LOAD_DATASET and not RUN_DOWNLOAD:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
else:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("HF_DATASETS_OFFLINE", None)

import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"project={PROJECT_ROOT}; device={DEVICE}; torch={torch.__version__}")


project=/Users/dani/Documents/uni/4t/erasmus/applied/project; device=mps; torch=2.15.0


## Dataset

Load the pinned Defactify revision from the local cache or download it when enabled. Check the required columns and the published split sizes: 42,000 train, 9,000 validation, and 45,000 test images.


In [5]:
import shutil
from datasets import load_dataset, DownloadConfig

raw = None
if RUN_LOAD_DATASET:
    if RUN_DOWNLOAD:
        free_gib = shutil.disk_usage(PROJECT_ROOT).free / 2**30
        if free_gib < 20:
            raise RuntimeError(
                f"Only {free_gib:.1f} GiB free. Free at least 20 GiB before full download/cache preparation."
            )
    try:
        raw = load_dataset(
            DATASET_ID,
            name="default",
            revision=DATASET_REVISION,
            cache_dir=str(HF_HOME / "datasets"),
            download_config=DownloadConfig(local_files_only=not RUN_DOWNLOAD),
        )
    except (FileNotFoundError, OSError, ValueError) as exc:
        if not RUN_DOWNLOAD:
            raise RuntimeError(
                "The complete prepared dataset is not available in data/huggingface/. "
                "For the first download, restart the kernel and set RUN_DOWNLOAD=True."
            ) from exc
        raise
    sizes = {split: len(raw[split]) for split in raw}
    assert sizes == EXPECTED_SIZES, f"Unexpected published split sizes: {sizes}"
    assert sum(sizes.values()) == 96_000
    assert {"Caption", "Image", "Label_A", "Label_B"}.issubset(
        raw["train"].column_names
    )
    if not RUN_DOWNLOAD:
        cache_paths = [
            entry["filename"] for split in raw.values() for entry in split.cache_files
        ]
        assert cache_paths and all(
            DATASET_REVISION in path for path in cache_paths
        ), "The local cache does not match the pinned dataset revision."
    print(
        "Loaded",
        sizes,
        "from",
        "download/cache" if RUN_DOWNLOAD else "local cache (offline)",
    )
else:
    print(
        "Dataset loading disabled. Set RUN_LOAD_DATASET=True to use the prepared local cache."
    )


Using the latest cached version of the dataset since Rajarshi-Roy-research/Defactify_Image_Dataset couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at /Users/dani/Documents/uni/4t/erasmus/applied/project/data/huggingface/datasets/Rajarshi-Roy-research___defactify_image_dataset/default/0.0.0/1c3b714c9691eb6dd11c2c1e92eab1ac29584fa0 (last modified on Fri Nov 13 11:20:04 2026).


Loaded {'validation': 9000, 'train': 42000, 'test': 45000} from local cache (offline)


## Data checks and leakage-safe splits

Check labels, image readability, and cross-split overlap in normalized captions and exact image hashes. If overlap is found, keep linked records together in deterministic replacement splits and audit them again, preserving all 96,000 images.


In [6]:
from defactify_splits import audit_dataset, make_leakage_safe_splits

published_raw = raw
published_audit = audit = split_report = None
if RUN_AUDIT:
    if raw is None:
        raise RuntimeError("Load the complete dataset first (RUN_LOAD_DATASET=True).")
    published_audit, records = audit_dataset(raw, EXPECTED_SIZES, SOURCE_NAMES)
    print("Published split audit:")
    display(published_audit["counts"], published_audit["formats_by_source"])
    print(
        {
            k: v
            for k, v in published_audit.items()
            if k not in ("counts", "formats_by_source")
        }
    )
    assert not published_audit["invalid_rows"], "Invalid labels or unreadable images"
    assert sum(published_audit["counts"]["count"]) == 96_000
    source_totals = published_audit["counts"].groupby("source")["count"].sum().to_dict()
    assert source_totals == {source: 16_000 for source in SOURCE_NAMES}, source_totals

    if (
        published_audit["caption_overlap_groups"]
        or published_audit["byte_identical_overlap_groups"]
    ):
        raw, split_report = make_leakage_safe_splits(
            published_raw, records, EXPECTED_SIZES, SEED
        )
        print("Replacement split:", split_report)
        audit, _ = audit_dataset(raw, EXPECTED_SIZES, SOURCE_NAMES)
        del _
    else:
        audit = published_audit
        print("Published split is leakage-safe; no replacement needed.")
    del records
    print("Training/evaluation split audit:")
    display(audit["counts"])
    print({k: v for k, v in audit.items() if k not in ("counts", "formats_by_source")})
    assert not audit[
        "invalid_rows"
    ], "Invalid labels or unreadable images after resplitting"
    assert (
        audit["caption_overlap_groups"] == 0
    ), "Caption overlap remains after resplitting"
    assert (
        audit["byte_identical_overlap_groups"] == 0
    ), "Identical images remain after resplitting"
    assert sum(len(raw[split]) for split in EXPECTED_SIZES) == 96_000
    assert all(len(raw[split]) > 0 for split in EXPECTED_SIZES)
    assert audit["counts"].groupby("source")["count"].sum().to_dict() == source_totals
else:
    print("Full audit disabled; enable RUN_AUDIT before training.")


Published split audit:


,split,binary,source,count
0,test,0,0,7500
1,test,1,1,7500
2,test,1,2,7500
3,test,1,3,7500
4,test,1,4,7500
5,test,1,5,7500
6,train,0,0,7000
7,train,1,1,7000
8,train,1,2,7000
9,train,1,3,7000


,source,format,count
0,real,JPEG,16000
1,SD 2.1,JPEG,16000
2,SDXL,JPEG,16000
3,SD 3,JPEG,16000
4,DALL-E 3,JPEG,16000
5,Midjourney v6,JPEG,16000


{'caption_overlap_groups': 60, 'caption_overlap_rows': 1758, 'byte_identical_overlap_groups': 12, 'byte_identical_overlap_rows': 54, 'invalid_rows': [], 'unique_dimensions_by_source': {'real': 842, 'SD 2.1': 1, 'SDXL': 1, 'SD 3': 1, 'DALL-E 3': 2, 'Midjourney v6': 2}}
Replacement split: {'connected_groups': 9826, 'moved_rows': 642, 'moves': {'train->test': 402, 'train->validation': 12, 'validation->test': 72, 'validation->train': 156}, 'split_sizes': {'train': 41742, 'validation': 8784, 'test': 45474}, 'split_fingerprint_sha256': 'cb673604921c4e2dee12a9e2f6a4a64a784e9c915726acc0a109281b22ec3255'}
Training/evaluation split audit:


,split,binary,source,count
0,test,0,0,7579
1,test,1,1,7579
2,test,1,2,7579
3,test,1,3,7579
4,test,1,4,7579
5,test,1,5,7579
6,train,0,0,6957
7,train,1,1,6957
8,train,1,2,6957
9,train,1,3,6957


{'caption_overlap_groups': 0, 'caption_overlap_rows': 0, 'byte_identical_overlap_groups': 0, 'byte_identical_overlap_rows': 0, 'invalid_rows': [], 'unique_dimensions_by_source': {'real': 842, 'SD 2.1': 1, 'SDXL': 1, 'SD 3': 1, 'DALL-E 3': 2, 'Midjourney v6': 2}}


## Image processing

Convert images to RGB tensors resized to 224 × 224, without augmentation. Build a DataLoader for each audited split, shuffling training data only.


In [7]:
from torch.utils.data import Dataset, DataLoader
from defactify_models import make_image_transform

image_transform = make_image_transform(IMAGE_SIZE)


class DefactifyTorchDataset(Dataset):
    def __init__(self, hf_split):
        self.rows = hf_split

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        return (
            image_transform(row["Image"].convert("RGB")),
            torch.tensor(int(row["Label_A"]), dtype=torch.float32),
            torch.tensor(int(row["Label_B"]), dtype=torch.long),
        )


def make_loaders(dataset):
    return {
        split: DataLoader(
            DefactifyTorchDataset(dataset[split]),
            batch_size=BATCH_SIZE,
            shuffle=(split == "train"),
            num_workers=0,
            pin_memory=(DEVICE.type == "cuda"),
        )
        for split in EXPECTED_SIZES
    }


loaders = make_loaders(raw) if raw is not None and audit is not None else None
if loaders is not None:
    print(
        "DataLoaders ready from audited replacement splits:",
        {split: len(loader.dataset) for split, loader in loaders.items()},
    )


DataLoaders ready from audited replacement splits: {'train': 41742, 'validation': 8784, 'test': 45474}


## Models

Build the RGB ResNet-18 baseline and the RGB/Fourier models with concatenation or attention plus a six-class auxiliary source head. Check their output shapes with synthetic inputs without downloading pretrained weights.


In [8]:
from defactify_models import build_detector

# Architecture smoke test needs no dataset and downloads no pretrained weights.
with torch.no_grad():
    sample = torch.rand(2, 3, IMAGE_SIZE, IMAGE_SIZE)
    for name, config in (
        ("rgb_baseline", {}),
        ("rgb_fourier_concat", {"fusion": "concat", "auxiliary": False}),
        ("rgb_fourier_attention_aux", {"fusion": "attention", "auxiliary": True}),
    ):
        binary, source = build_detector(name, config, pretrained=False).eval()(sample)
        assert binary.shape == (2,)
        assert source is None or source.shape == (2, 6)
        print(
            f"Model check: {name}: binary logits {tuple(binary.shape)}"
            + (f", source logits {tuple(source.shape)}" if source is not None else "")
        )


Model check: rgb_baseline: binary logits (2,)
Model check: rgb_fourier_concat: binary logits (2,)
Model check: rgb_fourier_attention_aux: binary logits (2,), source logits (2, 6)


## Training and validation

Train the three models with a binary loss weighted using training class counts; the auxiliary model also predicts the source label. Select checkpoints by validation balanced accuracy and save epoch metrics, the best weights, and the latest training state.


In [9]:
from defactify_training import binary_metrics, train_experiment

CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
EXPERIMENT_CONFIGS = {
    "rgb_baseline": {},
    "rgb_fourier_concat": {"fusion": "concat", "auxiliary": False},
    "rgb_fourier_attention_aux": {"fusion": "attention", "auxiliary": True},
}
training_metadata = {
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "seed": SEED,
    "split_sizes": (
        {split: len(raw[split]) for split in EXPECTED_SIZES}
        if raw is not None
        else None
    ),
    "image_size": IMAGE_SIZE,
    "source_names": SOURCE_NAMES,
    "split_report": split_report,
    "post_repair_audit": (
        {
            "caption_overlap_groups": audit["caption_overlap_groups"],
            "byte_identical_overlap_groups": audit["byte_identical_overlap_groups"],
            "invalid_row_count": len(audit["invalid_rows"]),
        }
        if audit is not None
        else None
    ),
}
trained = {}
if RUN_TRAIN:
    if loaders is None:
        raise RuntimeError("Run the complete leakage audit before training.")
    print(
        "Training on the repaired full dataset; checkpoints and CSVs are saved after every epoch."
    )
    for name, config in EXPERIMENT_CONFIGS.items():
        has_checkpoint = (CHECKPOINTS_DIR / name / "last.pt").exists()
        model = build_detector(name, config, pretrained=not has_checkpoint)
        model, history, best_score = train_experiment(
            model,
            name,
            config,
            loaders["train"],
            loaders["validation"],
            DEVICE,
            EPOCHS,
            LEARNING_RATE,
            AUX_LOSS_WEIGHT,
            training_metadata,
            CHECKPOINTS_DIR,
            OUTPUTS_DIR,
            resume=True,
        )
        trained[name] = {
            "model": model,
            "history": history,
            "best_validation_balanced_accuracy": best_score,
        }
    print("All experiments completed. Model selection uses validation only:")
    for name, record in trained.items():
        print(
            f"  {name}: best validation BA={record['best_validation_balanced_accuracy']:.4f}"
        )
else:
    print(
        "Training disabled. Existing checkpoints, if any, can still be exported below."
    )


Training on the repaired full dataset; checkpoints and CSVs are saved after every epoch.
[rgb_baseline] device=mps, train=41,742, validation=8,784, epochs=5, batch=32, lr=0.0001, pos_weight=0.200
[rgb_baseline] 1/5 (585.7s) train loss=0.0758, BA=0.9047; val loss=0.1045, BA=0.8701, F1=0.8583, AUROC=0.9772, BA gap=+0.0346  [new best]
[rgb_baseline] 2/5 (579.4s) train loss=0.0373, BA=0.9573; val loss=0.0702, BA=0.9181, F1=0.9289, AUROC=0.9832, BA gap=+0.0392  [new best]
[rgb_baseline] 3/5 (577.1s) train loss=0.0285, BA=0.9686; val loss=0.0390, BA=0.9577, F1=0.9762, AUROC=0.9919, BA gap=+0.0109  [new best]
[rgb_baseline] 4/5 (585.5s) train loss=0.0187, BA=0.9796; val loss=0.0683, BA=0.9396, F1=0.9792, AUROC=0.9893, BA gap=+0.0400
[rgb_baseline] 5/5 (580.7s) train loss=0.0193, BA=0.9775; val loss=0.0608, BA=0.9493, F1=0.9712, AUROC=0.9868, BA gap=+0.0282
[rgb_baseline] best epoch=3, validation BA=0.9577; CSV=/Users/dani/Documents/uni/4t/erasmus/applied/project/outputs/epochs_rgb_baseline.cs

## Export and final evaluation

Export the model selected by validation with its weights and inference configuration, then verify that it loads. When enabled, evaluate it on test and save binary metrics and recall by source; the cell refuses to overwrite an existing test result.


In [10]:
import json
from defactify_training import export_final_model
from defactify_models import load_detector


def evaluate_final(model, loader):
    model.eval()
    labels, sources, logits = [], [], []
    with torch.no_grad():
        for images, batch_labels, batch_sources in loader:
            batch_logits, _ = model(images.to(DEVICE))
            labels.extend(batch_labels.tolist())
            sources.extend(batch_sources.tolist())
            logits.extend(batch_logits.cpu().tolist())
    result = binary_metrics(labels, logits)
    predictions = (np.asarray(logits) >= 0).astype(int)
    result["recall_by_source"] = {
        SOURCE_NAMES[source]: float(
            np.mean(predictions[np.asarray(sources) == source] == int(source != 0))
        )
        for source in SOURCE_NAMES
        if np.any(np.asarray(sources) == source)
    }
    return result


selected_name = final_bundle_path = final_metadata = loaded_final = None
if RUN_EXPORT_FINAL:
    selected_name, final_bundle_path, final_metadata = export_final_model(
        CHECKPOINTS_DIR, EXPERIMENT_CONFIGS, EPOCHS, OUTPUTS_DIR, SOURCE_NAMES
    )
    loaded_final = load_detector(final_bundle_path, DEVICE)
    print(f"Verified exported bundle loads for inference: {loaded_final.model_name}")
else:
    print("Final-model export disabled; enable after experiments finish.")

if RUN_FINAL_TEST:
    if final_bundle_path is None or loaders is None:
        raise RuntimeError(
            "Export a selected model and load the audited dataset before final test."
        )
    metrics_path = OUTPUTS_DIR / "final_test_metrics.json"
    if metrics_path.exists():
        raise RuntimeError(
            f"Final test already recorded at {metrics_path}; do not repeat model selection on test."
        )
    final_model = loaded_final.model
    final_result = evaluate_final(final_model, loaders["test"])
    metrics_path.write_text(
        json.dumps({"model_name": selected_name, "metrics": final_result}, indent=2)
        + "\n"
    )
    print(
        f"Final test for {selected_name}: BA={final_result['balanced_accuracy']:.4f}, "
        f"F1={final_result['f1_ai']:.4f}, AUROC={final_result['auroc']:.4f}"
    )
    print("Confusion matrix [real, AI]:", final_result["confusion_matrix_real_ai"])
    print("Recall by source:", final_result["recall_by_source"])
    print("Saved one-time final metrics to", metrics_path)
else:
    print("Final test remains sealed (RUN_FINAL_TEST=False).")


Final model selected using validation only: rgb_fourier_attention_aux, BA=0.9658
Frontend bundle: /Users/dani/Documents/uni/4t/erasmus/applied/project/outputs/final_model.pt; raw weights: /Users/dani/Documents/uni/4t/erasmus/applied/project/outputs/final_weights.pt; metadata: /Users/dani/Documents/uni/4t/erasmus/applied/project/outputs/final_model.json
Verified exported bundle loads for inference: rgb_fourier_attention_aux
Final test for rgb_fourier_attention_aux: BA=0.8808, F1=0.8760, AUROC=0.9734
Confusion matrix [real, AI]: [[7420, 159], [8236, 29659]]
Recall by source: {'real': 0.9790209790209791, 'SD 2.1': 0.6938910146457317, 'SDXL': 0.8344108721467212, 'SD 3': 0.7473281435545587, 'DALL-E 3': 0.8039319171394643, 'Midjourney v6': 0.8337511545058715}
Saved one-time final metrics to /Users/dani/Documents/uni/4t/erasmus/applied/project/outputs/final_test_metrics.json


## Results comparison

Read the saved epoch logs and final metrics to compare training progress, model validation scores, validation–test performance, and recall by source. This cell does not load the dataset or run models.


In [ ]:
# Artifact-only comparison: safe to rerun without loading Defactify.
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

artifact_root = Path.cwd()
while not (artifact_root / "README.md").exists() and artifact_root != artifact_root.parent:
    artifact_root = artifact_root.parent
output_dir = artifact_root / "outputs"
model_names = ("rgb_baseline", "rgb_fourier_concat", "rgb_fourier_attention_aux")
epochs = {
    name: pd.read_csv(output_dir / f"epochs_{name}.csv") for name in model_names
}
model_info = json.loads((output_dir / "final_model.json").read_text())
test_info = json.loads((output_dir / "final_test_metrics.json").read_text())
assert test_info["model_name"] == model_info["model_name"]

epoch_table = pd.concat(
    [frame.assign(model=name)[["model", "epoch", "train_balanced_accuracy", "validation_balanced_accuracy", "validation_auroc"]]
     for name, frame in epochs.items()],
    ignore_index=True,
)
print("Evolution by epoch (look for divergence or unstable validation):")
display(epoch_table.round(4))

best_table = pd.DataFrame([
    {"model": name,
     "best_epoch": int(row["epoch"]),
     "validation_balanced_accuracy": row["validation_balanced_accuracy"],
     "validation_auroc": row["validation_auroc"]}
    for name, frame in epochs.items()
    for row in [frame.loc[frame["validation_balanced_accuracy"].idxmax()]]
]).sort_values("validation_balanced_accuracy", ascending=False)
print("Model selection used validation only:")
display(best_table.round(4))

selected = epochs[model_info["model_name"]]
selected_best = selected.loc[selected["validation_balanced_accuracy"].idxmax()]
val_cm = json.loads(selected_best["validation_confusion_matrix_real_ai"])
test = test_info["metrics"]
test_cm = test["confusion_matrix_real_ai"]
comparison = pd.DataFrame({
    "split": ["validation", "test"],
    "balanced_accuracy": [selected_best["validation_balanced_accuracy"], test["balanced_accuracy"]],
    "balanced_error_rate": [selected_best["validation_balanced_error_rate"], test["balanced_error_rate"]],
    "auroc": [selected_best["validation_auroc"], test["auroc"]],
    "real_recall": [val_cm[0][0] / sum(val_cm[0]), test_cm[0][0] / sum(test_cm[0])],
    "ai_recall": [val_cm[1][1] / sum(val_cm[1]), test_cm[1][1] / sum(test_cm[1])],
    "false_real_as_ai": [val_cm[0][1], test_cm[0][1]],
    "missed_ai_as_real": [val_cm[1][0], test_cm[1][0]],
})
print("Selected model: same fixed 0.5 probability threshold in both splits:")
display(comparison.round(4))
print("Balanced-accuracy gap:", round(comparison.loc[0, "balanced_accuracy"] - comparison.loc[1, "balanced_accuracy"], 4))

# The repaired test contains 7,579 records per source (see DATA_AUDIT_REPORT.md).
per_source = test_info["metrics"]["recall_by_source"]
source_count = model_info["training_metadata"]["split_sizes"]["test"] // len(per_source)
source_table = pd.DataFrame([
    {"source": name, "test_count": source_count, "recall": recall,
     "incorrect": round(source_count * (1 - recall))}
    for name, recall in per_source.items()
]).sort_values("recall")
print("Test recall by source; incorrect counts use the audited equal per-source split:")
display(source_table.round(4))
print("Ordinary test accuracy:", round((test_cm[0][0] + test_cm[1][1]) / sum(map(sum, test_cm)), 4))
print("Always-AI ordinary accuracy:", round(sum(test_cm[1]) / sum(map(sum, test_cm)), 4))


## Validation diagnostics

Load the saved detector and measure validation recall by source. Compare the default threshold with one selected by validation balanced accuracy, saving the diagnostics without modifying the exported detector or evaluating test.


In [ ]:
RUN_VALIDATION_DIAGNOSTICS = False  # set True only after loading and auditing the cached dataset above
if RUN_VALIDATION_DIAGNOSTICS:
    if loaders is None:
        raise RuntimeError("Load and audit Defactify before validation diagnostics")
    from defactify_models import load_detector
    from defactify_training import collect_predictions, select_balanced_threshold, source_recall

    saved = load_detector(output_dir / "final_model.pt", DEVICE)
    val_labels, val_sources, val_logits = collect_predictions(saved.model, loaders["validation"], DEVICE)
    threshold_study = select_balanced_threshold(val_labels, val_logits)
    source_study = source_recall(val_labels, val_sources, val_logits)
    print("Validation-only threshold study (exploratory, NOT a new test result):")
    display(pd.DataFrame([threshold_study]).round(4))
    display(pd.DataFrame([
        {"source": SOURCE_NAMES[source], **values}
        for source, values in source_study.items()
    ]).sort_values("recall").round(4))
    (output_dir / "validation_diagnostics.json").write_text(json.dumps({
        "model_name": saved.model_name,
        "split": "validation",
        "threshold_study": threshold_study,
        "source_recall_at_original_threshold": source_study,
    }, indent=2) + "\n")
else:
    print("Validation diagnostics disabled; the artifact-only tables above remain available.")


## References

- [Defactify dataset card](https://huggingface.co/datasets/Rajarshi-Roy-research/Defactify_Image_Dataset/blob/1c3b714c9691eb6dd11c2c1e92eab1ac29584fa0/README.md)
- [Data audit and split protocol](DATA_AUDIT_REPORT.md)
- [Experiment results](RESULTS.md)
